# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/tools/mlcroissant/index.html) library, which interfaces seamlessly with datasets described in the [Croissant schema](https://mlcommons.github.io/croissant/).

### Dataset Source
The dataset is described by a Croissant schema available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

It consists of a tabular cohort (N=77) of second primary colorectal cancer survivors, including extensive clinical and molecular variables.

In [ ]:
# Ensure 'mlcroissant' is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and accessible records using `mlcroissant`. This will instantiate an `mlcroissant.Dataset` object from the Croissant schema URL so that further exploration and processing can be performed.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description:\n{metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Explore the available record sets, fields, columns, and their respective `@id`s. Only `@id` references should be used for subsequent data extraction.

Let's enumerate each record set, and inspect their fields and columns.

In [ ]:
# Discover available record sets, fields, columns in the dataset schema
record_sets = list(dataset.record_sets)

print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '<no name>')}")
    # List fields if present
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        print(f"  Fields:")
        for f in fields:
            print(f"    - @id: {f['@id']} | name: {f.get('name', '<no name>')} | dataType: {f.get('dataType', '<unknown>')}")
    # List columns if present
    if 'column' in rs:
        cols = rs['column']
        if not isinstance(cols, list):
            cols = [cols]
        print(f"  Columns:")
        for col in cols:
            print(f"    - @id: {col['@id']} | name: {col.get('name', '<no name>')}")

## 3. Data Extraction
We load the tabular data from the primary record set using its `@id`. Please reference all entities (record sets, fields, columns) **by their `@id`**.

First, we'll collect all tabular record set `@id`s, then demonstrate loading records from one of them into a Pandas DataFrame.

In [ ]:
# Collect all available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("RecordSet @id list:")
for i, r in enumerate(record_set_ids):
    print(f"  [{i}] {r}")

# We'll use the first record set for demonstration (edit as needed)
target_record_set_id = record_set_ids[0]

records = list(dataset.records(record_set=target_record_set_id))
df = pd.DataFrame(records)

print(f"\nLoaded {len(df)} records from RecordSet '{target_record_set_id}'.")
print(f"DataFrame columns:\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Here we perform basic exploratory analysis (filtering, normalization, grouping) on relevant numeric or categorical fields.

**Note:** Adjust the field `@id`s as needed based on output from the previous overview. Below, we use example field identifiers and logic that you should update according to your actual dataset structure and content.

In [ ]:
# Find numeric fields by inspecting the schema
numeric_fields = []
categorical_fields = []
for rs in dataset.record_sets:
    if rs['@id'] == target_record_set_id:
        for f in rs.get('field', []):
            dt = f.get('dataType', '').lower()
            # Heuristic for numeric
            if dt in ['integer', 'float', 'number']:
                numeric_fields.append(f['@id'])
            elif dt in ['text', 'string', 'category', 'boolean']:
                categorical_fields.append(f['@id'])
print(f"Numeric field @ids: {numeric_fields}")
print(f"Categorical field @ids: {categorical_fields}")

# Pick a numeric field for demonstration
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")
else:
    print("No numeric fields found. Please review the schema.")
    numeric_field_id = None

threshold = 10  # Adjust threshold as meaningful for your field
if numeric_field_id and numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records: {len(filtered_df)} rows with {numeric_field_id} > {threshold}")
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Optionally group by a categorical field
    if categorical_fields:
        group_field = categorical_fields[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
            print(grouped_df)
else:
    print("Cannot filter and normalize without a valid numeric field.")

## 5. Visualization
Visualize distributions and relationships from the dataset using Matplotlib or Seaborn. Consider histograms for numeric fields and barplots for group means.

*You may need to modify the field `@id`s based on your schema exploration above.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Example: Histogram of a numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

# Example: Mean of numeric field grouped by a categorical field
if numeric_field_id and categorical_fields:
    group_field = categorical_fields[0]
    if group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.barplot(x=group_field, y=numeric_field_id, data=df, ci=None)
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to:
- Load and inspect a clinical cohort released in Croissant format
- Explore record sets, fields, and column structure using `@id` references
- Load tabular data for analysis and visualization
- Apply standard EDA steps: filtering, normalization, grouping, and plotting

**Key takeaways:** The FAIR² dataset is structured for interoperable access, enabling automated loading, schema inspection, and reproducible data analysis workflows. Adjust and extend this notebook for deeper insights specific to your research goals.